# Notebook 02 — Zero-shot Text2Cypher with Ollama

**Zero-shot** = we give the model the schema, some rules and the question, but **no solved examples**. This is the baseline every improvement in notebook 03 is measured against.

**Learning objectives**
* understand the anatomy of a Text2Cypher prompt (role, schema, domain knowledge, output rules, question),
* call a local model through Ollama with reproducible settings,
* extract a clean query from a chatty answer,
* observe the typical failure modes on our graph.

### Step 1 — Configuration and connection
**What it does:** same settings as notebook 01; we import the graph wrapper and schema extraction from `t2c.py` (they were explained in notebook 01) and rebuild the schema text.

In [1]:
import os, sys, json, re, time, math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --- Connection settings (override with environment variables if you prefer) ---
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")     # <- change me
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None          # None = default database

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL       = os.getenv("T2C_MODEL", "qwen2.5-coder:7b")      # any chat model you pulled with `ollama pull`
EMBED_MODEL = os.getenv("T2C_EMBED_MODEL", "nomic-embed-text")

print("Neo4j :", NEO4J_URI, "| Ollama:", OLLAMA_HOST, "| model:", MODEL)

Neo4j : bolt://localhost:7687 | Ollama: http://localhost:11434 | model: qwen2.5:7b


In [2]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)

from t2c import Neo4jGraph, extract_schema, schema_to_text

graph = Neo4jGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
schema_text = schema_to_text(extract_schema(graph))
print("Connected. Schema description:", len(schema_text), "characters")

Connected. Schema description: 5371 characters


### Step 2 — Domain knowledge and output rules
**What it does:** defines two blocks of text that complement the automatic schema.

* `DOMAIN_NOTES` explains **business semantics** the schema cannot express: the claim path, the meaning of `VERIFIED`, how to resolve skill aliases, what "meets a requirement" means, the location rule, `endDate = null`, today's date…
* `OUTPUT_RULES` constrains the **form** of the answer: one read-only query, only schema elements, literal values, return IDs + names, answer inside a ```` ```cypher ```` block.

**Why:** in practice most Text2Cypher errors are not syntax errors but *semantic* ones (the query runs and returns the wrong thing). Writing the rules once in the prompt is the cheapest fix. Rule 4 (return IDs and names) also makes automatic evaluation possible in notebook 04.

In [25]:
DOMAIN_NOTES = """\
Business rules and conventions of the TalentBridge graph:
- A candidate's skills are NOT linked directly to Skill. Path: (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill).
  SkillClaim.level: 1 = beginner, 2 = intermediate, 3 = advanced.
  SkillClaim.verificationStatus: 'VERIFIED', 'UNVERIFIED' (not checked yet) or 'REFUTED' (checked and false).
- Skill names have aliases. Aliases plays the role of alternative labels or synonyms. To find a skill from user text use:
  MATCH (:Alias {key: toLower('<text>')})-[:ALIAS_OF]->(s:Skill)
  Never use CONTAINS or STARTS WITH on skill names ('Java' is not 'JavaScript').
- Job requirements: (j:Job)-[r:REQUIRES_SKILL]->(s:Skill), r.priority is 'MANDATORY' or 'PREFERRED', r.minLevel is an integer.
- A mandatory requirement is MET when the candidate has a VERIFIED claim on that skill with level >= minLevel.
  A candidate meets all mandatory requirements when the number of MET mandatory requirements equals the number of mandatory requirements.
- Location rule: Job.workMode is 'REMOTE', 'HYBRID' or 'ONSITE'. A REMOTE job needs Candidate.remoteAvailable = true;
  HYBRID/ONSITE jobs need (c)-[:AVAILABLE_ONSITE_IN]->(city) where (j)-[:BASED_IN]->(city).
- Job.status is 'OPEN' or 'CLOSED'.
- Work history: (c:Candidate)-[:HAS_EXPERIENCE]->(e:WorkExperience)-[:AT_ORGANIZATION]->(o:Organization).
  WorkExperience.endDate is null when the job is ongoing. Dates are DATE values: compare with date('2025-01-01').
- Applications: (c:Candidate)-[:SUBMITTED]->(a:Application)-[:FOR_JOB]->(j:Job), (a)-[:HAS_EVENT]->(ev:ApplicationEvent).
  ev.type is one of 'APPLIED','WITHDRAWN','REAPPLIED','REVIEWED','REJECTED','REINSTATED'; ev.at is a DATETIME. Order history by ev.at.
- Skill assessments over time: (sa:SkillAssessment)-[:ASSESSES]->(cl:SkillClaim); sa.assessedAt is a DATE (null when unknown).
- Two different candidates can have the same name: identify people by candidateId.
- Today's date for this project is 2026-11-12.
"""

OUTPUT_RULES = """\
Rules for your answer:
1. Write exactly ONE read-only Cypher query for Neo4j 5. Never use CREATE, MERGE, SET, DELETE, REMOVE, DROP, LOAD CSV or CALL.
2. Use only the labels, relationship types, directions and properties listed in the schema.
3. Put literal values directly in the query (no $parameters).
4. When you return candidates, return c.candidateId and c.name; for jobs return j.jobId and j.title; for skills return s.name.
5. Return the Cypher inside a ```cypher code block and nothing else.
"""

### Step 3 — Assembling the system prompt
**What it does:** `build_system_prompt` concatenates a role statement, the schema, the domain notes and the rules.
**Why a *system* message:** chat models treat the system message as standing instructions and the user message as the task; keeping the (long, constant) context in the system message also lets Ollama reuse its cache between questions.

In [18]:
def build_system_prompt(schema_text: str, notes: str = DOMAIN_NOTES, rules: str = OUTPUT_RULES) -> str:
    return (
        "You are an expert Neo4j developer. You translate questions about the TalentBridge recruiting "
        "graph into Cypher.\n\n"
        f"### Graph schema\n{schema_text}\n\n"
        f"### Domain knowledge\n{notes}\n"
        f"### {rules}"
    )

system_prompt = build_system_prompt(schema_text)
print(system_prompt[:1500], "\n[...]")
print(f"\nTotal ≈ {len(system_prompt)//4} tokens")

You are an expert Neo4j developer. You translate questions about the TalentBridge recruiting graph into Cypher.

### Graph schema
Node labels and properties:
(:Alias {key: STRING e.g. 'python', 'python 3', label: STRING e.g. 'Python', 'Python 3'})
(:Application {applicationId: STRING e.g. 'APP-C01-J01', 'APP-C02-J01'})
(:ApplicationEvent {at: DATETIME e.g. '2026-09-01T09:00:00.000000000+00:00', '2026-09-02T10:30:00.000000000+00:00', attemptNo: INTEGER one of [1, 2], decision: STRING one of ['REINSTATE', 'REJECT', 'SHORTLIST'], eventId: STRING e.g. 'AEV01', 'AEV02', note: STRING e.g. 'Kubernetes evidence unclear', 'Verified K8s claim found on E11', seq: INTEGER one of [1, 2, 3], type: STRING one of ['APPLIED', 'REAPPLIED', 'REINSTATED', 'REJECTED', 'REVIEWED', 'WITHDRAWN']})
(:Assessor {assessorId: STRING e.g. 'A02', 'A01', kind: STRING one of ['TECHNICAL_PANEL'], name: STRING e.g. 'Yassine Amrani', 'Leila Mansouri', recruiterId: STRING e.g. 'R02'})
(:Candidate {candidateId: STRING e.g.

### Step 4 — The Ollama wrapper
**What it does:** `OllamaLLM` is a callable: `llm(messages) -> text`.
Two options matter a lot:
* `temperature = 0` → (almost) deterministic output: the same question gives the same query, which we need for debugging and evaluation;
* `num_ctx = 8192` → the context window. **Ollama's default (2048–4096 tokens depending on the version) is smaller than our prompt**: the beginning of the prompt (the schema!) would be silently cut off. This is the #1 "mysterious" bug students hit with Ollama.

`embed()` will be used in notebook 03 for example selection.

In [19]:
class OllamaLLM:
    """Callable wrapper: llm(messages) -> text. temperature 0 for reproducible Cypher."""

    def __init__(self, model: str = "qwen2.5-coder:7b", host: str | None = None,
                 temperature: float = 0.0, num_ctx: int = 8192):
        from ollama import Client
        self.client = Client(host=host or os.getenv("OLLAMA_HOST", "http://localhost:11434"))
        self.model = model
        self.options = {"temperature": temperature, "num_ctx": num_ctx}

    def __call__(self, messages: list[dict], fmt=None) -> str:
        kwargs = {"format": fmt} if fmt is not None else {}
        response = self.client.chat(model=self.model, messages=messages, options=self.options, **kwargs)
        return response["message"]["content"]

    def embed(self, texts: list[str], model: str = "nomic-embed-text") -> list[list[float]]:
        return self.client.embed(model=model, input=texts)["embeddings"]

llm = OllamaLLM(MODEL, host=OLLAMA_HOST, temperature=0.0, num_ctx=8192)

### Step 5 — Getting the query out of the answer
**What it does:** models rarely answer with *only* the query. `extract_cypher` takes the content of the first ```` ```cypher ```` fence; if there is none, it starts at the first Cypher keyword (`MATCH`, `WITH`, …). `strip_think` removes the hidden reasoning of "thinking" models.
**Why:** sending "Sure! Here is the query: MATCH…" to Neo4j is a syntax error that has nothing to do with the model's Cypher skill.

In [20]:
def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks produced by reasoning models (qwen3, deepseek-r1...)."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

CYPHER_START = '\\b(MATCH|OPTIONAL\\s+MATCH|WITH|UNWIND|RETURN)\\b'

def extract_cypher(text: str) -> str:
    """Pull the Cypher query out of an LLM answer (code fence first, then first Cypher keyword)."""
    text = strip_think(text)
    fence = re.search(r"```(?:cypher|sql)?\s*\n?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        query = fence.group(1)
    else:
        start = re.search(CYPHER_START, text, flags=re.IGNORECASE)
        query = text[start.start():] if start else text
    return query.strip().rstrip(";").strip()

# quick self-tests
print(extract_cypher("Sure!\n```cypher\nMATCH (n) RETURN n;\n```"))
print(extract_cypher("<think>let me think about MATCH...</think>Answer: MATCH (j:Job) RETURN j.title"))

MATCH (n) RETURN n
MATCH (j:Job) RETURN j.title


### Step 6 — Zero-shot messages and a helper to ask questions
**What it does:** `build_zero_shot_messages` creates `[system, user]`. `ask()` runs the whole loop once: prompt → LLM → extract → execute in a read transaction → show the query and the result table (or the error).

In [21]:
def build_zero_shot_messages(question: str, system_prompt: str) -> list[dict]:
    return [{"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: {question}"}]

def ask(question, prompt=None, show=True):
    """One zero-shot round trip. Returns (cypher, rows_or_None, error_or_None)."""
    t0 = time.time()
    raw = llm(build_zero_shot_messages(question, prompt or system_prompt))
    cypher = extract_cypher(raw)
    try:
        rows, error = graph.run_read(cypher), None
    except Exception as exc:
        rows, error = None, f"{type(exc).__name__}: {exc}"
    if show:
        print(f"Q: {question}   ({time.time()-t0:.1f}s)\n{cypher}\n")
        display(pd.DataFrame(rows)) if rows is not None else print("❌", error[:400])
    return cypher, rows, error

### Step 7 — An easy question
**Expected:** a one-pattern query on `Job.status = 'OPEN'` returning 6 jobs (J01, J02, J03, J05, J06, J07).
Easy questions mostly test whether the model respects the enumeration values (`'OPEN'`, not `'open'`).

In [26]:
_ = ask("List all open jobs with their titles.")

Q: List all open jobs with their titles.   (10.3s)
MATCH (j:Job {status: 'OPEN'})
RETURN j.jobId, j.title



,j.jobId,j.title
0,J01,Graph Data Engineer
1,J02,ML Engineer
2,J03,Data Analyst
3,J05,Big Data Engineer
4,J06,Platform Engineer
5,J07,Semantic Web Engineer


### Step 8 — A multi-hop question
**Expected:** the model must go through the claim node: `(c)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s)`, and filter `verificationStatus = 'VERIFIED'` and `level = 3`. Correct answer: **C01, C03, C05, C09**.
**Typical mistake:** `(c:Candidate)-[:HAS_SKILL]->(s:Skill)` — a relationship that does not exist. Neo4j returns **zero rows without an error** (it only warns). Ask the students why this is more dangerous than a syntax error.

In [27]:
_ = ask("Which candidates have a verified advanced-level Python claim?")

Q: Which candidates have a verified advanced-level Python claim?   (1.6s)
MATCH (c:Candidate)-[:MAKES_CLAIM]->(sc:SkillClaim {level: 3, verificationStatus: 'VERIFIED'})-[:ABOUT_SKILL]->(s:Skill {name: 'Python'})
RETURN c.candidateId, c.name, s.name



,c.candidateId,c.name,s.name
0,C01,Amina Idrissi,Python
1,C03,Sara Haddad,Python
2,C05,Lina Faris,Python
3,C09,Nadia Bennani,Python


### Step 9 — The alias trap
**Expected:** "ML" must be resolved through `(:Alias {key: 'ml'})-[:ALIAS_OF]->(s)`; correct answer **C03 (Sara Haddad)**.
Then "Java" must return **C04 and C10 only** — a query using `CONTAINS 'Java'` would wrongly add the JavaScript people (C04, C06 and C09's refuted claim).

In [30]:
_ = ask("Which candidates claim ML?")
_ = ask("Which candidates claim Java?")

Q: Which candidates claim ML?   (3.5s)
MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill {name: 'ML'})
RETURN c.candidateId, c.name, s.name



""


Q: Which candidates claim Java?   (1.1s)
MATCH (c:Candidate)-[:MAKES_CLAIM]->(sc:SkillClaim)-[:ABOUT_SKILL]->(s:Skill {name: 'Java'})
RETURN c.candidateId, c.name, s.name



,c.candidateId,c.name,s.name
0,C04,Omar Naji,Java
1,C10,Hamza Tazi,Java


### Step 10 — Ablation: what do the domain notes buy us?
**What it does:** asks the same questions with a prompt that contains the schema **only** (no business rules).
**What to observe:** without the notes the model typically writes `{name: 'ML'}` (no such skill → empty result) or `CONTAINS 'Java'` (wrong result). Same model, same schema; only the documentation changed.

In [13]:
schema_only_prompt = build_system_prompt(schema_text, notes="")
for q in ["Which candidates claim ML?", "Which candidates claim Java?"]:
    _ = ask(q, prompt=schema_only_prompt)

Q: Which candidates claim ML?   (3.6s)
MATCH (c:Candidate)-[:MAKES_CLAIM]->(sc:SkillClaim)-[:ABOUT_SKILL]->(s:Skill {name: 'ML'})
RETURN c.candidateId, c.name



""


Q: Which candidates claim Java?   (1.2s)
MATCH (c:Candidate)-[:MAKES_CLAIM]->(sc:SkillClaim)-[:ABOUT_SKILL]->(s:Skill {name: 'Java'})
RETURN c.candidateId, c.name, s.name



,c.candidateId,c.name,s.name
0,C04,Omar Naji,Java
1,C10,Hamza Tazi,Java


### Step 11 — A business-rule question
**Expected answer:** only **C01** meets all mandatory requirements of J01 (Python ≥2, Neo4j ≥2, Cypher ≥2, all verified). C08 meets 2 of 3, C05's Neo4j claim is unverified.
This requires counting met requirements and comparing with the number of mandatory requirements — a pattern 7B models often get wrong zero-shot (e.g. returning everyone who meets *any* requirement). Keep the result: notebook 03 fixes it with examples.

In [15]:
_ = ask("Which candidates meet all mandatory skill requirements of job J01?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: REQUIRES_SKILL)", position=<SummaryInputPosition line=3, column=60, offset=190>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'column': 60, 'offset': 190, 'line': 3}}> for query: "MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill),\n      (j:Job {jobId: 'J01'})-[:REQUIRES_SKILL]->(s)\nWHERE cl.verificationStatus = 'VERIFIED' AND cl.level >= j.REQUIRES_SKILL.minL

Q: Which candidates meet all mandatory skill requirements of job J01?   (2.9s)
MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill),
      (j:Job {jobId: 'J01'})-[:REQUIRES_SKILL]->(s)
WHERE cl.verificationStatus = 'VERIFIED' AND cl.level >= j.REQUIRES_SKILL.minLevel
WITH c, COUNT(DISTINCT s) AS metRequirements
MATCH (j)-[:REQUIRES_SKILL]->(s)
WITH c, metRequirements, COUNT(DISTINCT s) AS totalRequirements
WHERE metRequirements = totalRequirements
RETURN c.candidateId, c.name



""


### Step 12 — Temperature and reproducibility
**What it does:** asks the same question 3 times with `temperature = 0.8`.
**What to observe:** different queries (sometimes different answers!) for the same question. For code generation we keep temperature at 0; sampling several queries can still be useful for *self-consistency* voting (an exercise below).

In [ ]:
hot_llm = OllamaLLM(MODEL, host=OLLAMA_HOST, temperature=0.8)
q = "Which candidates can work onsite in Casablanca?"
for i in range(3):
    print(f"--- sample {i+1}\n", extract_cypher(hot_llm(build_zero_shot_messages(q, system_prompt))))

### Step 13 — Try your own questions
Some ideas, from easy to hard:
* "Which jobs are based in Rabat?"
* "Which companies has Karim Alaoui worked for, and in which roles?"
* "Show Youssef Karim's application events for the Graph Data Engineer job in chronological order."
* "Which skills does Amina Idrissi claim?" ← two different people have this name!
* "What was candidate C01's assessed Python level as of 1 January 2025?"

In [16]:
_ = ask("Which skills does Amina Idrissi claim?")

Q: Which skills does Amina Idrissi claim?   (3.6s)
MATCH (c:Candidate {name: 'Amina Idrissi'})-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill)
RETURN c.candidateId AS candidateId, c.name AS name, s.name AS skillName



,candidateId,name,skillName
0,C01,Amina Idrissi,Python
1,C01,Amina Idrissi,Neo4j
2,C01,Amina Idrissi,Cypher
3,C01,Amina Idrissi,Docker
4,C07,Amina Idrissi,SPARQL
5,C07,Amina Idrissi,TypeScript
6,C07,Amina Idrissi,RDF


## Failure-mode taxonomy (to discuss with the class)

| Type | Example on our graph | Detected by Neo4j? | Fix (notebook) |
|---|---|---|---|
| Syntax error | missing parenthesis, `GROUP BY` (SQL habit) | ✅ error | self-correction (03) |
| Hallucinated schema | `(:Candidate)-[:HAS_SKILL]->(:Skill)` | ⚠️ only a warning, 0 rows | schema in prompt (01), few-shot (03) |
| Wrong direction | `(s:Skill)-[:ABOUT_SKILL]->(cl)` | ⚠️ 0 rows | explicit patterns in schema |
| Wrong literal | `verificationStatus = 'verified'` | ⚠️ 0 rows | enumerations in schema (01) |
| Entity error | `{name: 'ML'}`, `CONTAINS 'Java'` | ❌ wrong rows | domain notes (02), entity linking (03) |
| Business-rule error | "meets requirements" ignoring verification | ❌ wrong rows | domain notes + examples (03), trusted templates (05) |
| Ambiguity | "Amina Idrissi" = C01 or C07 | ❌ silently mixes two people | entity linking (03) |

**Key message:** the dangerous errors are the ones that *execute successfully*. That is why notebook 04 measures **execution accuracy** against gold answers, not just "did it run".

## Exercises
1. Add a rule to `DOMAIN_NOTES` that fixes one error you observed, and re-run the question. Did it break another question?
2. Implement *self-consistency*: sample 5 queries at temperature 0.8, execute them, and return the most frequent result.
3. Compare two models (`MODEL = "llama3.1:8b"`) on Steps 7–11. Which errors are model-specific?